# Pyrefact — Automated Python Code Refactoring
Install: `pip install pyrefact` | CLI: `python -m pyrefact <file>`

In [9]:
import pyrefact
print(dir(pyrefact))

['__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__main__', '__name__', '__package__', '__path__', '__spec__', 'abstractions', 'compile', 'constants', 'core', 'find_replace', 'findall', 'finditer', 'fix', 'fixes', 'format_code', 'format_file', 'format_files', 'formatting', 'fullmatch', 'logs', 'main', 'match', 'object_oriented', 'parsing', 'pattern_matching', 'performance', 'performance_numpy', 'performance_pandas', 'processing', 'search', 'style', 'sub', 'subn', 'symbolic_math', 'tracing']


In [10]:
import pkgutil, importlib
for finder, name, ispkg in pkgutil.walk_packages(pyrefact.__path__, pyrefact.__name__ + '.'):
    try:
        mod = importlib.import_module(name)
        print(name, '->', dir(mod))
    except Exception as e:
        print(name, '-> ERROR:', e)

pyrefact.__main__ -> ['__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__spec__', 'main', 'sys']
pyrefact.abstractions -> ['Collection', 'Iterable', 'Sequence', 'Tuple', '_EverythingContainer', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__spec__', '_build_function_body', '_definite_external_effects', '_definite_stored_names', '_get_constant_insertion_lineno', '_get_function_insertion_lineno', '_group_nodes_by_purpose', '_hashable_node_purpose_type', '_possible_external_effects', '_scoped_dependencies', 'ast', 'collections', 'constants', 'core', 'create_abstractions', 'hash_node', 'itertools', 'overused_constant', 'parsing', 'processing', 're', 'simplify_if_control_flow', 'style', 'tracing']
pyrefact.constants -> ['ASSUMED_PACKAGES', 'ASSUMED_SOURCES', 'AST_TYPES_WITH_BODY', 'AST_TYPES_WITH_ORELSE', 'BUILTIN_FUNCTIONS', 'COMPARISON_OPERATORS', 'ITERATOR_FUNCTIONS', 'MATH_FUNCTIONS', 'Mappin

In [11]:
# Raw pyrefact CLI output — diff between original and refactored
import subprocess, tempfile, os, shutil

original = '''
import os
import os

x = 1 + 0
y = x * 1

items = [1, 2, 3, 4, 5]
result = []
for item in items:
    result.append(item * 2)

def get_name():
    name = "World"
    return name

idx = 0
for item in items:
    print(idx, item)
    idx += 1
'''

with tempfile.NamedTemporaryFile(mode='w', suffix='.py', delete=False) as f:
    f.write(original)
    tmp = f.name

backup = tmp + '.bak'
shutil.copy(tmp, backup)

r = subprocess.run(['python', '-m', 'pyrefact', tmp], capture_output=True, text=True, timeout=120)
print('STDOUT:')
print(r.stdout)
print('STDERR:')
print(r.stderr)
print('RETURNCODE:', r.returncode)

# Show raw unified diff
diff = subprocess.run(['python', '-c',
    f'import difflib; a=open(r"{backup}").readlines(); b=open(r"{tmp}").readlines(); '
    f'print("".join(difflib.unified_diff(a, b, fromfile="original", tofile="refactored")))'
], capture_output=True, text=True)
print('--- UNIFIED DIFF ---')
print(diff.stdout)

os.unlink(tmp)
os.unlink(backup)

STDOUT:

STDERR:

RETURNCODE: 0
--- UNIFIED DIFF ---
--- original
+++ refactored
@@ -1,20 +1,10 @@
 
-import os
-import os
 
-x = 1 + 0
-y = x * 1
 
-items = [1, 2, 3, 4, 5]
-result = []
-for item in items:
-    result.append(item * 2)
+ITEMS = [1, 2, 3, 4, 5]
 
-def get_name():
-    name = "World"
-    return name
 
-idx = 0
-for item in items:
-    print(idx, item)
-    idx += 1
+IDX = 0
+for item in ITEMS:
+    print(IDX, item)
+    IDX += 1




In [12]:
# Raw format_code() API output on a redditwarp file
import pyrefact, os, difflib

target = os.path.join(os.path.dirname(os.getcwd()), 'redditwarp', 'redditwarp', 'http', 'connector_ASYNC.py')
original_src = open(target, encoding='utf-8').read()

try:
    fixed = pyrefact.format_code(original_src)
    diff = list(difflib.unified_diff(
        original_src.splitlines(keepends=True),
        fixed.splitlines(keepends=True),
        fromfile='original',
        tofile='pyrefact'
    ))
    print(''.join(diff) if diff else '(no changes suggested)')
except Exception as e:
    print('ERROR:', e)

--- original
+++ pyrefact
@@ -1,7 +1,3 @@
 
 from __future__ import annotations
 
-from .handler_ASYNC import Handler
-
-class Connector(Handler):
-    pass



In [13]:
# Raw pyrefact findall — AST pattern matches
import pyrefact, os, json

src = open(os.path.join(os.path.dirname(os.getcwd()), 'redditwarp', 'redditwarp', 'util', 'base_conversion.py'), encoding='utf-8').read()

# Find all append-in-loop patterns
patterns = [
    ('list_append_loop', 'for $X in $Y:\n    $Z.append($W)'),
    ('x_times_one', '$X * 1'),
    ('x_plus_zero', '$X + 0'),
]

for label, pattern in patterns:
    try:
        matches = list(pyrefact.findall(pattern, src))
        print(f'{label}: {len(matches)} match(es)')
        for m in matches:
            print(' ', repr(m))
    except Exception as e:
        print(f'{label}: ERROR - {e}')

list_append_loop: ERROR - invalid syntax (<unknown>, line 1)
x_times_one: ERROR - invalid syntax (<unknown>, line 1)
x_plus_zero: ERROR - invalid syntax (<unknown>, line 1)
